# Preparación de Dataset para Modelos de IA
**Programa:** SENA - Procesamiento de Datos para Modelos de Inteligencia Artificial 
**Aprendiz:** Melissa López Díaz  
**Fecha:** 06 de Mayo 2026  

### Objetivo
Preparar el dataset de clientes aplicando codificación de variables categóricas,
escalamiento de variables numéricas y separación train/test para entrenamiento de modelos de IA.

## 1. Diagnóstico y diccionario de variables (4, 6 y 8 mayo)

Del dato limpio al dataset preparado para IA

In [27]:
# Importación de librería Pandas

import pandas as pd


In [28]:
#Carga del Dataset

df = pd.read_csv('../data/raw/dataset_clientes_limpio_para_preprocesamiento.csv')

In [29]:
# Diagnóstico técnico del dataset

print(df.shape) # Dimensiones del dataset
print(df.dtypes) # Tipos de datos por variable
df.head() # Primeras filas

(260, 13)
ID_Cliente             str
Edad                 int64
IngresoMensual       int64
CantidadCompras      int64
Ciudad                 str
Genero                 str
Segmento               str
NivelSatisfaccion      str
AntiguedadMeses      int64
DiasUltimaCompra     int64
GastoPromedio        int64
CanalPreferido         str
Compra               int64
dtype: object


,ID_Cliente,Edad,IngresoMensual,CantidadCompras,Ciudad,Genero,Segmento,NivelSatisfaccion,AntiguedadMeses,DiasUltimaCompra,GastoPromedio,CanalPreferido,Compra
0,C1000,22,2235000,7,Bucaramanga,Femenino,Básico,Bajo,14,102,90000,WhatsApp,0
1,C1001,55,1965000,6,Medellín,Masculino,Premium,Bajo,61,90,70000,App móvil,0
2,C1002,49,2501000,4,Bogotá,Masculino,Medio,Medio,38,100,320000,App móvil,0
3,C1003,39,1604000,4,Bucaramanga,Femenino,Medio,Bajo,34,23,130000,App móvil,1
4,C1004,38,3967000,4,Cartagena,Masculino,Básico,Alto,52,172,509000,App móvil,0


### Diagnóstico inicial (hallazgos)

- El dataset contiene 260 registros correspondientes a clientes distribuidos en 13 variables. 

- Las variables categóricas son: ID_Cliente (identificadora), Ciudad, Genero, Segmento, NivelSatisfaccion y CanalPreferido.

- Las variables numéricas son: Edad, IngresoMensual, CantidadCompras, AntiguedadMeses, DiasUltimaCompra y GastoPromedio.

- La variable objetivo es Compra, una variable binaria que indica si el cliente realizó una 
compra (`1`) o no (`0`).

In [30]:
df.describe() 

,Edad,IngresoMensual,CantidadCompras,AntiguedadMeses,DiasUltimaCompra,GastoPromedio,Compra
count,260.000000,2.600000e+02,260.000000,260.000000,260.000000,2.600000e+02,260.000000
mean,41.534615,3.633192e+06,6.019231,34.057692,89.042308,3.686885e+05,0.261538
std,13.097008,3.447769e+06,2.223865,20.380562,52.046405,3.297269e+05,0.440320
min,18.000000,9.390000e+05,1.000000,1.000000,2.000000,4.500000e+04,0.000000
25%,31.000000,2.171250e+06,4.000000,18.000000,47.000000,1.710000e+05,0.000000
50%,41.000000,2.988500e+06,6.000000,33.000000,85.000000,2.955000e+05,0.000000
75%,53.000000,4.022750e+06,7.000000,52.000000,137.000000,4.807500e+05,1.000000
max,65.000000,4.539500e+07,14.000000,71.000000,180.000000,3.675000e+06,1.000000


### Interpretación del Diagnóstico Técnico

El análisis estadístico sobre los 260 registros confirma que las variables numéricas
presentan comportamientos muy distintos entre sí, lo que justifica el uso de
diferentes técnicas de escalamiento:

- Edad oscila entre 18 y 65 años con un promedio de 41.5 años y una desviación
estándar de 13. Distribución estable y manejable. Se aplicará StandardScaler.

- CantidadCompras varía entre 1 y 14 compras con promedio de 6 y desviación
estándar de 2.2. Sin valores extremos evidentes. Se aplicará StandardScaler.

- AntiguedadMeses va de 1 a 71 meses con promedio de 34 y desviación estándar
de 20. Distribución uniforme. Se aplicará StandardScaler.

- DiasUltimaCompra oscila entre 2 y 180 días con promedio de 89 y desviación
estándar de 52. Rango amplio pero sin valores extremos que distorsionen el escalado.
Se aplicará StandardScaler.

- IngresoMensual tiene un promedio de 3.6 millones pero un máximo de 45.3 millones
con una desviación estándar de 3.4 millones, casi igual al promedio. Esto confirma
la presencia de valores extremos que afectarían un StandardScaler. Se aplicará RobustScaler.

- GastoPromedio promedia 368 mil pesos con un máximo de 3.67 millones y una
desviación estándar de 329 mil, también cercana al promedio. Misma situación que
IngresoMensual. Se aplicará RobustScaler.

- Compra es la variable objetivo binaria. El promedio de 0.26 indica que
aproximadamente el 26% de los clientes realizó una compra (1) y el 74% no compró (0).
Esto refleja un dataset moderadamente desbalanceado, aspecto a considerar al
entrenar y evaluar el modelo. No se transforma.

### Sesión 1 - 4 mayo: ¿Qué significa preparar datos para IA?

Preparar datos para IA significa transformar el dataset para que el modelo pueda procesarlo matemáticamente.

Se obtuvo un dataset de clientes con variables numéricas, nominales, ordinales, identificadoras y una variable objetivo, este dataset llegó limpio, sin valores nulos, sin errores, es decir que tiene valores correctos y formatos consistentes. Como primer paso realicé un diagnóstico inicial, es decir una exploración en el dataset, para posteriormente hacer la clasificación de cada variable y definír su tratamiento. Se  presentan cuatro problemas que un modelo no puede resolver solo:

1. Identifiqué variables categóricas en texto como Ciudad, Genero, Segmento, CanalPreferido. El modelo no lee palabras, necesita números. Por eso aplicaré la técnica One-Hot Encoding.

2. Identifiqué a NivelSatisfaccion como variable ordinal con jerarquía la cuál tiene un orden de Bajo < Medio < Alto. No se puede convertir al azar, hay que respetar esa jerarquía. Por eso aplicaré OrdinalEncoder.

3. Identifiqué variables numéricas en escalas incompatibles como Edad e IngresoMensual/GastoPromedio. Si no las escalo el modelo va a creer que el ingreso importa miles de veces más que la edad, solo por el tamaño del número. Por eso usaré StandardScaler y RobustScaler.

4. Se excluirá ID_Cliente, este es un identificador pero que no aporta nada y es solo una etiqueta. No tiene relación con si alguien compra o no. Si lo dejo el modelo intentará aprender de él y aprenderá basura.

## Tabla Inicial de Variables

| # | Variable | Tipo de Dato | Categoría | Valores Ejemplo |
|---|---|---|---|---|
| 1 | ID_Cliente | str | Identificadora | C1000, C1001, C1002 |
| 2 | Edad | int64 | Numérica discreta | 22, 55, 49 |
| 3 | IngresoMensual | int64 | Numérica continua | 2.235.000, 1.965.000 |
| 4 | CantidadCompras | int64 | Numérica discreta | 7, 6, 4 |
| 5 | Ciudad | str | Categórica nominal | Bogotá, Medellín, Cartagena |
| 6 | Genero | str | Categórica nominal | Femenino, Masculino |
| 7 | Segmento | str | Categórica nominal | Básico, Medio, Premium |
| 8 | NivelSatisfaccion | str | Categórica ordinal | Bajo, Medio, Alto |
| 9 | AntiguedadMeses | int64 | Numérica discreta | 14, 61, 38 |
| 10 | DiasUltimaCompra | int64 | Numérica discreta | 102, 90, 100 |
| 11 | GastoPromedio | int64 | Numérica continua | 90.000, 70.000, 320.000 |
| 12 | CanalPreferido | str | Categórica nominal | WhatsApp, App móvil |
| 13 | Compra | int64 | Binaria objetivo | 0, 1 |

### Sesión 2 - 6 mayo: Tipos de variables y decisiones de preprocesamiento

En esta sesión analicé cada variable del dataset para clasificarla según su tipo
y definir qué técnica de preprocesamiento aplicarle. Este paso es fundamental porque
no todas las variables se transforman igual, la decisión depende de la naturaleza
del dato, su escala y el rol que cumple dentro del modelo. A partir de este análisis
construí el diccionario de variables que guiará todo el proceso de preparación del dataset.

## Diccionario de Variables

### Variables identificadora a excluir y objetivo
| Variable | Tipo | Descripción | Tratamiento |
|---|---|---|---|
| ID_Cliente | Categórica Identificadora | Identificador único del cliente | Excluir del modelo |
| Compra | Binaria objetivo | Indica si el cliente compró: 1=Sí, 0=No | No transformar |

### Variables categóricas
| Variable | Tipo | Descripción | Tratamiento |
|---|---|---|---|
| Ciudad | Nominal | Ciudad principal del cliente | One-Hot Encoding |
| Genero | Nominal | Género reportado por el cliente | One-Hot Encoding |
| Segmento | Nominal | Segmento comercial del cliente | One-Hot Encoding |
| CanalPreferido | Nominal | Canal principal de interacción | One-Hot Encoding |
| NivelSatisfaccion | Ordinal | Satisfacción del cliente: Bajo < Medio < Alto | OrdinalEncoder: Bajo=0, Medio=1, Alto=2 |

### Variables numéricas
| Variable | Tipo | Descripción | Tratamiento |
|---|---|---|---|
| Edad | Discreta | Edad del cliente en años | StandardScaler |
| CantidadCompras | Discreta | Número histórico de compras  | StandardScaler |
| AntiguedadMeses | Discreta | Meses de antiguedad del cliente | StandardScaler |
| DiasUltimaCompra | Discreta | Días desde la última compra | StandardScaler |
| IngresoMensual | Continua | Ingreso mensual en pesos colombianos | RobustScaler |
| GastoPromedio | Continua | Gasto promedio por compra en pesos colombianos | RobustScaler |

### Diccionario de variables

El diccionario de variables muestra la clasificación según su naturaleza: identificadora, objetivo, categóricas nominales y ordinal, por último las numéricas. Se detalla cada variable, su tipo, descripción y se identifica el tratamiento adecuado para cada variable.

El objetivo del preprocesamiento es transformar las variables categóricas a formato numérico y escalar las variables numéricas para que el modelo de IA pueda procesarlas correctamente. La variable objetivo no se transforma por ser la variable que el modelo debe predecir y el identificador se debe excluir. 

### ¿Cómo preparar técnicamente un dataset limpio para que pueda ser utilizado en el entrenamiento y evaluación de un modelo de inteligencia artificial, aplicando codificación de variables categóricas, escalamiento de variables numéricas y separación adecuada de los datos?

Cuando recibí el dataset de clientes lo primero que noté es que aunque estaba limpio, no estaba listo para un modelo de inteligencia artificial. Contiene un identificador de cliente, una variable objetivo binaria, variables escritas en texto y los modelos de IA no entienden palabras sino solo números. También tiene variables numéricas en escalas o rangos muy diferentes e incompatibles como la edad y el ingreso mensual y posibles valores extremos que no se pueden comparar directamente sin una transformación previa.

Para resolver esto primero he identificado cada variable, la clasifiqué según su tipo y definí qué técnica aplicar en cada caso. Las variables categóricas nominales como Ciudad, Genero, Segmento, CanalPreferido las codificaré con One-Hot Encoding, La variable ordinal NivelSatisfaccion la trataré con OrdinalEncoder respetando su jerarquía natural como Bajo, Medio y Alto. Para las variables numéricas usaré StandardScaler en las que tenían distribución normal y RobustScaler en las que tenían valores extremos como IngresoMensual y GastoPromedio.

Finalmente separaré el dataset en entrenamiento y prueba antes de aplicar cualquier transformación para evitar la fuga de información. El resultado será dos archivos listos para ser usados por un modelo, uno para entrenarlo y otro para evaluarlo.

### Sesión 3 - 8 mayo: Diagnóstico técnico antes de codificar

## Diagnóstico Técnico - Antes de Codificar

### Dimensiones
El dataset contiene 260 registros y 13 variables, de las cuales 6 son numéricas, 
5 son categóricas, 1 es identificadora y 1 es la variable objetivo.

### Variables numéricas - Análisis de rangos
A partir del análisis estadístico se identificaron los siguientes comportamientos:

- Edad presenta valores entre 18 y 65 años con un promedio de 41.5 años,
lo que indica una distribución relativamente uniforme entre adultos. Se aplicará StandardScaler.

- IngresoMensual tiene un promedio de 3.63 millones pero un máximo de 45.39 millones,
con una desviación estándar muy alta. Esto indica la presencia de valores extremos
que distorsionarían un StandardScaler. Se aplicará RobustScaler.

- CantidadCompras presenta valores entre 1 y 14 con un promedio de 6 compras.
Distribución estable sin valores extremos. Se aplicará StandardScaler.

- AntiguedadMeses varía entre 1 y 71 meses con promedio de 34 meses.
Distribución manejable. Se aplicará StandardScaler.

- DiasUltimaCompra oscila entre 2 y 180 días con un promedio de 89 días
y una desviación estándar de 52, lo que representa un rango amplio pero
sin valores extremos que distorsionen significativamente el escalado.
Se aplicará StandardScaler.

- GastoPromedio tiene un máximo de 3.67 millones con alta desviación estándar,
lo que confirma la presencia de valores extremos. Se aplicará RobustScaler.

### Variables categóricas - Valores únicos
Se verificaron los valores únicos de cada variable categórica para confirmar
que no hay categorías inesperadas ni errores de escritura antes de codificar.

### Variable objetivo
- Compra es una variable binaria: 26% de los clientes compró (1) y 74% no compró (0).
Esto indica un dataset moderadamente desbalanceado, lo cual deberá considerarse
al momento de entrenar el modelo.

### Columnas a transformar
- Excluir: ID_Cliente
- One-Hot Encoding: Ciudad, Genero, Segmento, CanalPreferido
- OrdinalEncoder: NivelSatisfaccion (Bajo=0, Medio=1, Alto=2)
- StandardScaler: Edad, CantidadCompras, AntiguedadMeses, DiasUltimaCompra
- RobustScaler: IngresoMensual, GastoPromedio
- Sin transformar: Compra